# Proyecto RappiPlus: de datos a decisiones de negocio

**Introducción**


El objetivo de este proyecto es evaluar el desempeño del servicio **RappiPlus** para apoyar **decisiones de negocio basadas en datos**.

Se trabajan con múltiples datasets del negocio:

- **rappiplus_orders_raw.csv** → información de pedidos, precios, descuentos y revenue  
- **rappiplus_catalog.csv** → costos de productos, categorías y proveedores  
- **rappiplus_marketing_spend.csv** → inversión en marketing por canal y país  
- **events / users / user_activity (SQL)** → comportamiento del usuario dentro de la plataforma  
- **experiment_checkout_ui.csv** → resultados de un experimento A/B en el checkout  

El análisis sigue una lógica clara y progresiva:

1. 🔍 Evaluar si podemos confiar en los datos (calidad de datos en Python) 

2. 💰 Analizar si el negocio es rentable (revenue, costos y profit)  

3. 🛒 Entender dónde se pierden los usuarios (funnel de conversión)  

4. 🔁 Evaluar si los usuarios regresan (retención por cohortes)  

5. 🧪 Validar si los cambios generan impacto (test estadístico)  

6. 📊 Comunicar los resultados (dashboard en BI)  

A lo largo del proyecto, se transforman datos en insights para responder preguntas clave del negocio y proponer **recomendaciones accionables**.

---

## 🔹 Paso 1: Cargar y validar la calidad de los datos

---

### 1.1 Carga de datos y vista rápida

**🎯 Objetivo:** Familiarizarte con la estructura de los datasets del negocio antes de analizarlos.

**Instrucciones:**

- Importa las librerías necesarias
- Carga los archivos:
  - `rappiplus_orders_raw.csv`
  - `rappiplus_catalog.csv`
  - `rappiplus_marketing_spend.csv`
- Guarda los DataFrames en:
  - `orders`, `catalog`, `marketing`
- Explora cada dataset.

---

In [1]:
# importar librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy as stats

ModuleNotFoundError: No module named 'seaborn'

In [ ]:
# cargar archivos
orders = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_orders_raw.csv')
catalog = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_catalog.csv')
marketing = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_marketing_spend.csv')

In [ ]:
# explorar datasets
orders.head(5)

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
0,order_0,user_6993,2025-05-22,Argentina,desktop,organic,Jacket-Winter-M,Moda,2.0,332.69,0.0,665.37
1,order_1,user_1329,2025-06-15,Mexico,desktop,paid_search,Tablet-Standard-64GB,Electronica,1.0,176.86,5.0,171.86
2,order_2,user_3194,2025-05-02,Argentina,desktop,social,Blender-XL-Red,Hogar,2.0,102.99,10.0,195.99
3,order_3,user_4510,2025-06-09,Colombia,mobile,social,Tablet-Standard-64GB,Electronica,1.0,257.87,15.0,242.87
4,order_4,user_5044,2025-03-30,Argentina,desktop,paid_search,Blender-XL-Red,Hogar,1.0,336.28,0.0,336.28


In [ ]:
catalog.head(5)

,nombre_producto,categoria_producto,costo_unitario,proveedor
0,Laptop-Gaming-16GB,Electrónica,280.68,"Fuller, Pena and Myers"
1,Phone-Pro-128GB,Electrónica,10.12,King Ltd
2,Tablet-Standard-64GB,Electrónica,25.21,Bowers LLC
3,Blender-XL-Red,Hogar,176.64,Long-Reid
4,Vacuum-Pro-Black,Hogar,16.60,"Rivera, Carr and Finley"


In [ ]:
marketing.head(5)

,fecha,pais,id_campaña,canal,gasto
0,2025-01-01,Mexico,organic_Mexico,organic,2446.25
1,2025-01-01,Mexico,paid_search_Mexico,paid_search,2704.34
2,2025-01-01,Mexico,social_Mexico,social,2045.01
3,2025-01-01,Colombia,organic_Colombia,organic,2597.21
4,2025-01-01,Colombia,paid_search_Colombia,paid_search,1771.40


---

### Revisión y calidad de datos

**🎯 Objetivo:** Detectar y corregir problemas en los datos que puedan afectar el análisis de revenue, costos y rentabilidad.

Se revisan los 3 datasets
- Validar y convertir fechas al formato correcto  
- Revisar variables numéricas (sin negativos o ceros inválidos)  
- Verificar consistencia de montos  
- Eliminar duplicados  
- Revisar variables categóricas 

---

### Valores nulos detectados

Se identificaron valores nulos en las siguientes columnas de `orders`:

| Columna            | % Nulos |
|--------------------|---------|
| pais               | 1.19%   |
| dispositivo        | 0.08%   |
| fuente_referencia  | 0.12%   |
| nombre_producto    | 0.12%   |
| categoria_producto | 0.32%   |
| cantidad           | 0.20%   |
| precio_unitario    | 0.20%   |
| monto_descuento    | 0.20%   |

**Decisión:** Se decide conservar los valores nulos ya que el porcentaje 
es muy bajo (menor al 2%) y no afecta significativamente el análisis.

In [ ]:
orders['fecha_hora_pedido'] = pd.to_datetime(orders['fecha_hora_pedido'], errors='coerce')

In [ ]:
marketing['fecha'] = pd.to_datetime(marketing['fecha'], errors='coerce')

In [ ]:
orders = orders[orders['cantidad'] > 0]

In [ ]:
orders['monto_calculado'] = (orders['precio_unitario'] * orders['cantidad']) - orders['monto_descuento']

In [ ]:
orders['diferencia'] = orders['monto_total'] != orders['monto_calculado']

In [ ]:
orders[orders['diferencia'] == True][['precio_unitario', 'cantidad', 
                                       'monto_descuento', 'monto_total', 
                                       'monto_calculado']].head(10)

,precio_unitario,cantidad,monto_descuento,monto_total,monto_calculado
0,332.69,2.0,0.0,665.37,665.38
2,102.99,2.0,10.0,195.99,195.98
6,163.59,2.0,0.0,327.19,327.18
7,373.68,2.0,0.0,747.35,747.36
18,180.64,2.0,5.0,356.27,356.28
20,489.11,2.0,10.0,968.23,968.22
21,203.26,2.0,0.0,406.51,406.52
23,487.85,2.0,5.0,970.71,970.70
24,117.99,2.0,0.0,235.99,235.98
35,467.34,2.0,5.0,929.69,929.68


In [ ]:
orders['diferencia_valor'] = abs(orders['monto_total'] - orders['monto_calculado'])
print(orders['diferencia_valor'].max())

0.010000000000104592


In [ ]:
orders = orders.drop(columns=['monto_calculado', 'diferencia', 'diferencia_valor'])

In [ ]:
print(f"Total filas: {len(orders)}")
print(f"IDs únicos: {orders['id_pedido'].nunique()}")

Total filas: 25046
IDs únicos: 24946


In [ ]:
orders = orders.drop_duplicates(subset=['id_pedido'], keep='first')

In [ ]:
orders['pais'] = orders['pais'].replace({'mexico': 'Mexico', 'colombia': 'Colombia', 'argentina': 'Argentina'})

In [ ]:
print(f"Total filas: {len(marketing)}")
print(f"Campañas únicas: {marketing['id_campaña'].nunique()}")

Total filas: 1620
Campañas únicas: 9


---
**📦 Exportación**: Una vez finalizada la limpieza, se exportan los datasets para utilizarlos en la última etapa del proyecto.

In [ ]:
# exportar datasets
orders.to_csv('orders_clean.csv', index=False)
catalog.to_csv('catalog_clean.csv', index=False)
marketing.to_csv('marketing_clean.csv', index=False)

---

## 🔹 Paso 2: Analizar si el negocio es rentable

### 2.1 Cálculo de KPIs principales

**🎯 Objetivo:** Calcular los indicadores clave del negocio para evaluar ingresos, costos y rentabilidad.

Se usan los 3 datasets (`orders`, `catalog`, `marketing_spend`):

**📊 Parte 1: Rentabilidad del negocio**
- ¿Cuál es el ingreso total (revenue)? 
- ¿Cuál es el costo total? 
- ¿Cuánto se ha invertido en marketing? 
- ¿El negocio es rentable? (calcular profit)  

---

**📈 Parte 2: Comportamiento de ventas**
- ¿Cuál es el ticket promedio por orden? 
- ¿Cuál es la cantidad promedio de productos por orden? 
- ¿Cuál es el producto más vendido?
- ¿Cuánto se ha gastado en marketing por canal? 

In [ ]:
ingreso_total = orders['monto_total'].sum()
print(f"ingreso_total: {ingreso_total:,.2f}")

print()

orders_catalog = orders.merge(catalog, on='nombre_producto', how='left')

print()

orders_catalog['costo_orden'] = orders_catalog['cantidad'] * orders_catalog['costo_unitario']

costo_total = orders_catalog['costo_orden'].sum()
print(f"Costo total: {costo_total:,.2f}")

ingreso_total: 51,966,981.56


Costo total: 43,124,069.01


In [ ]:
marketing_total = marketing['gasto'].sum()
print(f"Inversión en marketing: {marketing_total:,.2f}")

Inversión en marketing: 2,871,843.53


In [ ]:
profit = ingreso_total - costo_total - marketing_total
print(f"Profit: {profit:,.2f}")

Profit: 5,971,069.02


In [ ]:
ticket_promedio = orders['monto_total'].mean()
print(f"Ticket promedio: {ticket_promedio:,.2f}")

Ticket promedio: 2,083.18


In [ ]:
cantidad_promedio = orders['cantidad'].mean()
print(f"Cantidad promedio por orden: {cantidad_promedio:,.2f}")

Cantidad promedio por orden: 7.12


In [ ]:
producto_mas_vendido = orders.groupby('nombre_producto')['cantidad'].sum().sort_values(ascending=False)
print(producto_mas_vendido.head(5))

nombre_producto
Laptop-Gaming-16GB    144198.0
Vacuum-Pro-Black        6284.0
Blender-XL-Red          6279.0
Jacket-Winter-M         6256.0
Sneakers-Urban-42       6172.0
Name: cantidad, dtype: float64


In [ ]:
marketing_por_canal = marketing.groupby('canal')['gasto'].sum()
print(marketing_por_canal)

canal
organic        913533.01
paid_search    863088.21
social         918043.21
Name: gasto, dtype: float64


---

## 🔹 Paso 3: Entender dónde se pierden los usuarios (funnel de conversión)

**🎯 Objetivo:** Analizar el comportamiento de los usuarios para identificar en qué etapa del proceso se pierden.


⚙️**Conexión a la base de datos**:  
Se ejecuta la línea de configuración para conectar con la base de datos y aplicar consultas SQL en la tabla **events**.

---

**📊 Parte 1: Construcción del funnel**
- ¿Cuántos usuarios llegan a cada etapa del funnel?  
- Se calcula el número de usuarios únicos por `nombre_evento`  
- Se ordenan los eventos según el flujo del usuario  

---

**📉 Parte 2: Análisis de conversión**
- Se calcula la tasa de conversión entre cada paso del funnel  
- Se identifica en qué etapa se pierde la mayor cantidad de usuarios  
- ¿Cuál es la tasa de conversión final?
---

In [ ]:
import pandas as pd
from sqlalchemy import create_engine

# ======================
# Conexión (NO modificar)
# ======================
db_config = {
    'user': 'practicum_student',
    'pwd': 'QnmDH8Sc2TQLvy2G3Vvh7',
    'host': 'yp-trainers-practicum.cluster-czs0gxyx2d8w.us-east-1.rds.amazonaws.com',
    'port': 5432,
    'db': 'data-analyst-production-db-en'
}

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

engine = create_engine(connection_string, connect_args={'sslmode':'require'})

In [ ]:
# Explorar tabla events
# =========================
query_events = '''
SELECT *
FROM events;
'''
events = pd.read_sql(query_events, con=engine)
events.head()

,id_usuario,id_sesion,nombre_evento,timestamp_evento,pais,dispositivo,fuente_referencia,categoria_producto
0,user_6772,6a97f2af-32ae-4186-8c92-04025be1a27b,first_visit,2025-05-17,Colombia,desktop,organic,Moda
1,user_5883,369b767c-1c33-4b2f-a652-c7c0ef92cfc9,add_to_cart,2025-02-23,Mexico,mobile,social,Hogar
2,user_5946,60039041-e78b-474c-87b3-c0b7e9c30708,add_payment_info,2025-05-15,Colombia,desktop,social,Electronica
3,user_827,18252a64-f389-4ef7-9e58-dadad4a3491e,purchase,2025-03-31,Mexico,mobile,social,Moda
4,user_2361,221b364e-cdc5-4668-b698-18d5ba849a67,first_visit,2025-01-22,Argentina,desktop,paid_search,Electronica


In [ ]:
# PARTE 1: Totales del funnel
# ======================

query_totals = '''
SELECT 
    nombre_evento,
    COUNT(DISTINCT id_usuario) AS usuarios_unicos
FROM events
GROUP BY nombre_evento
ORDER BY 
    CASE nombre_evento
        WHEN 'first_visit' THEN 1
        WHEN 'select_item' THEN 2
        WHEN 'add_to_cart' THEN 3
        WHEN 'begin_checkout' THEN 4
        WHEN 'add_payment_info' THEN 5
        WHEN 'purchase' THEN 6
    END;
'''

totals = pd.read_sql(query_totals, con=engine)
totals

,nombre_evento,usuarios_unicos
0,first_visit,7796
1,select_item,7582
2,add_to_cart,7634
3,begin_checkout,7208
4,add_payment_info,6250
5,purchase,6240


In [ ]:
# PARTE 2: Conversiones
# ======================
query_conversion = '''
SELECT 
    nombre_evento,
    usuarios_unicos,
    ROUND(usuarios_unicos * 100.0 / LAG(usuarios_unicos) OVER (ORDER BY orden), 2) AS tasa_conversion
FROM (
    SELECT 
        nombre_evento,
        COUNT(DISTINCT id_usuario) AS usuarios_unicos,
        CASE nombre_evento
            WHEN 'first_visit' THEN 1
            WHEN 'select_item' THEN 2
            WHEN 'add_to_cart' THEN 3
            WHEN 'begin_checkout' THEN 4
            WHEN 'add_payment_info' THEN 5
            WHEN 'purchase' THEN 6
        END AS orden
    FROM events
    GROUP BY nombre_evento
) subquery
ORDER BY orden;
'''

conversion = pd.read_sql(query_conversion, con=engine)
conversion



,nombre_evento,usuarios_unicos,tasa_conversion
0,first_visit,7796,NaN
1,select_item,7582,97.26
2,add_to_cart,7634,100.69
3,begin_checkout,7208,94.42
4,add_payment_info,6250,86.71
5,purchase,6240,99.84


In [ ]:
tasa_final = (6240 / 7796) * 100
print(f"Tasa de conversión final: {tasa_final:.2f}%")

Tasa de conversión final: 80.04%


---

## 🔹 Paso 4: Evaluar si los usuarios regresan (retención por cohortes)

**🎯 Objetivo:** Analizar la retención de usuarios para entender si regresan después de registrarse.

**Tablas**

- `users` 
- `user_activity` 

---
1. Se identifica la cohorte de cada usuario según el **mes de registro**.


2. Se calcula la retención semanal: cuántos usuarios **se mantienen activos** en cada semana desde su registro.
   - `retenido_w1`: usuarios activos en la semana 1  
   - `retenido_w2`: usuarios activos en la semana 2  
   - `retenido_w3`: usuarios activos en la semana 3  


3. Se calcula el porcentaje de retención para cada semana, dividiendo los usuarios retenidos entre los clientes iniciales de la cohorte:  
   - `semana_1`: porcentaje de usuarios retenidos en la semana 1  
   - `semana_2`: porcentaje de usuarios retenidos en la semana 2  
   - `semana_3`: porcentaje de usuarios retenidos en la semana 3  

Se revisa que la columna de fecha esté en formato correcto (`DATE`).  
Se realiza la conversión usando: `CAST(fecha_registro AS DATE)`

In [ ]:
# Explorar tabla users
# =========================
query_users = '''
SELECT *
FROM users;
'''
users = pd.read_sql(query_users, con=engine)
users.head(3)

,id_usuario,fecha_registro,país,dispositivo,tipo_plan
0,user_0,2025-01-29,Mexico,mobile,free
1,user_1,2025-01-07,Mexico,mobile,free
2,user_2,2025-03-12,Argentina,mobile,free


In [ ]:
# Explorar tabla user_activity
# =========================
query_user_activity = '''
SELECT *
FROM user_activity
'''
user_activity = pd.read_sql(query_user_activity, con=engine)
user_activity.head(3)

,id_usuario,fecha_actividad,dias_despues_registro,activo
0,user_0,2025-02-05,7,0
1,user_0,2025-02-12,14,1
2,user_0,2025-02-19,21,1


In [ ]:
# Retención por cohortes
# ======================

query_cohort_retention_final = '''
SELECT
    cohorte,
    usuarios_totales,
    retenido_w1,
    retenido_w2,
    retenido_w3,
    ROUND(retenido_w1 * 100.0 / usuarios_totales, 2) AS semana_1,
    ROUND(retenido_w2 * 100.0 / usuarios_totales, 2) AS semana_2,
    ROUND(retenido_w3 * 100.0 / usuarios_totales, 2) AS semana_3
FROM (
    SELECT
        DATE_TRUNC('month', CAST(fecha_registro AS DATE)) AS cohorte,
        COUNT(DISTINCT u.id_usuario) AS usuarios_totales,
        SUM(CASE WHEN ua.dias_despues_registro = 7 AND ua.activo = 1 THEN 1 ELSE 0 END) AS retenido_w1,
        SUM(CASE WHEN ua.dias_despues_registro = 14 AND ua.activo = 1 THEN 1 ELSE 0 END) AS retenido_w2,
        SUM(CASE WHEN ua.dias_despues_registro = 21 AND ua.activo = 1 THEN 1 ELSE 0 END) AS retenido_w3
    FROM users u
    LEFT JOIN user_activity ua ON u.id_usuario = ua.id_usuario
    GROUP BY cohorte
) subquery
ORDER BY cohorte;
'''

# Ejecutar la consulta
cohorte_final = pd.read_sql(query_cohort_retention_final, con=engine)
cohorte_final

,cohorte,usuarios_totales,retenido_w1,retenido_w2,retenido_w3,semana_1,semana_2,semana_3
0,2025-01-01 00:00:00+00:00,1627,697,668,656,42.84,41.06,40.32
1,2025-02-01 00:00:00+00:00,1444,611,609,635,42.31,42.17,43.98
2,2025-03-01 00:00:00+00:00,1636,677,705,690,41.38,43.09,42.18
3,2025-04-01 00:00:00+00:00,1606,680,697,663,42.34,43.40,41.28
4,2025-05-01 00:00:00+00:00,1687,695,676,706,41.20,40.07,41.85


---

## 🔹 Paso 5: Validar si los cambios generan impacto (test estadístico)

🎯 **Objetivo:** Evaluar si la modificación en la UI del checkout impacta la **tasa de conversión de compra**.

---

1. **Analizar el dataset** `experiment_checkout_ui.csv` para identificar la métrica principal **conversion**.
   - La métrica **conversion** es 1 si el usuario completó la compra, 0 si no.    
2. **Plantear la hipótesis estadística**     
3. **Aplicar el test estadístico adecuado** 
4. **Interpretar el resultado**  

---
Hipótesis estadística
   - **H₀ (Hipótesis nula):** No existe diferencia significativa en la tasa de conversión entre la UI original (control) y la nueva UI (tratamiento)
   - **H₁ (Hipótesis alternativa):** La nueva UI (tratamiento) genera una mayor tasa de conversión que la UI original (control)
   
**Test estadístico:** La columna convirtio tiene valores 0 y 1 por lo que el test más adecuado es Chi-cuadrado

**Nivel de significancia alpha:** 0.05

**No implementar la nueva UI todavía**

1. Continuar el experimento con más usuarios
   para obtener resultados más concluyentes

2. Revisar si hay segmentos específicos donde 
   la nueva UI sí tenga impacto significativo
   (por país, dispositivo, etc.)

3. Considerar modificar el diseño de la nueva 
   UI para generar un impacto más pronunciado

In [ ]:
# Cargar dataset del experimento
experiment = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/experiment_checkout_ui.csv')
experiment.head(5)

,id_usuario,variante,convirtio,dispositivo,pais,duracion_sesion,timestamp
0,exp_user_0,tratamiento,0,mobile,Argentina,114.41,2025-03-28
1,exp_user_1,tratamiento,0,desktop,Mexico,170.03,2025-01-15
2,exp_user_2,control,1,mobile,Colombia,140.21,2025-03-18
3,exp_user_3,tratamiento,0,mobile,Colombia,151.45,2025-06-03
4,exp_user_4,tratamiento,0,desktop,Mexico,299.96,2025-01-12


In [ ]:
# Contar usuarios por variante
print(experiment['variante'].value_counts())
print()
# Tasa de conversión por variante
print(experiment.groupby('variante')['convirtio'].mean())

tratamiento    5035
control        4965
Name: variante, dtype: int64

variante
control        0.156898
tratamiento    0.162860
Name: convirtio, dtype: float64


In [ ]:
from scipy.stats import chi2_contingency

# Tabla de contingencia
tabla = pd.crosstab(experiment['variante'], experiment['convirtio'])
print(tabla)

convirtio       0    1
variante              
control      4186  779
tratamiento  4215  820


In [ ]:
chi2_stat, p_value, dof, expected = chi2_contingency(tabla)
print(f"Estadístico chi2: {chi2_stat:.4f}")
print(f"Valor p: {p_value:.4f}")

Estadístico chi2: 0.6178
Valor p: 0.4319


### Interpretación del resultado

Con un valor p = 0.4319 y un nivel de significancia alpha = 0.05:

**p_value (0.4319) > alpha (0.05) → No se rechaza H₀**

No existe evidencia estadísticamente significativa de que la nueva UI 
del checkout genere una mayor tasa de conversión que la UI original.

Aunque los datos muestran una diferencia leve:
- Control:     15.69%
- Tratamiento: 16.29%

Esta diferencia de +0.60% podría ser atribuida al azar y no 
al cambio en la UI del checkout.

---